### 1.(1)

记 $p(x_i; \beta) = P(Y=1|X=x_i) = \frac{\exp(x_i^T\beta)}{1+\exp(x_i^T\beta)}$
$$
\begin{aligned}
P(Y=y|X=x) &= \prod_{i=1}^{N} P(Y_i=y_i|X=x_i) \\
&= \prod_{i=1}^{N} p(x_i; \beta)^{y_i} (1 - p(x_i; \beta))^{1-y_i} \\
\end{aligned}\\
\begin{aligned}
l(\beta) &= \sum_{i=1}^N (y_i \log p(x_i; \beta) + (1-y_i) \log(1 - p(x_i; \beta))) \\
&= \sum_{i=1}^N (y_i x_i^T \beta - \log(1 + \exp(x_i^T \beta))) \\
\end{aligned}\\
\begin{aligned}
\frac{\partial l(\beta)}{\partial \beta^T} &= \sum_{i=1}^N (y_i x_i - \frac{\exp(x_i^T \beta)x_i}{1+\exp(x_i^T \beta)})\\
&= \sum_{i=1}^N (y_i - p(x_i; \beta)) x_i \\
&= X^T(y-p(X;\beta)) \quad \text{其中} p(X;\beta) = (p(x_1;\beta), p(x_2;\beta), \ldots, p(x_N;\beta))^T\\
\end{aligned}\\
\begin{aligned}
\frac{\partial^2l(\beta)}{\partial \beta \partial \beta^T} &= -\sum_{i=1}^N \frac{x_i \exp(x_i^T \beta)x_i^T(1+\exp(x_i^T \beta))-x_i \exp(x_i^T \beta)\exp(x_i^T\beta)x_i^T}{(1+\exp(x_i^T \beta))^2} \\
&= -\sum_{i=1}^N \frac{x_i x_i^T \exp(x_i^T \beta)}{(1+\exp(x_i^T \beta))^2}\\
&= -\sum_{i=1}^N x_i x_i^T p(x_i; \beta)(1-p(x_i; \beta)) \\
&= -X^T W X \quad \text{其中} W = \text{diag}(p(x_i; \beta)(1-p(x_i; \beta)))\\
\end{aligned}\\
\begin{aligned}
\beta^{new} &= \beta^{old} - \left(\frac{\partial^2 l(\beta)}{\partial \beta \partial \beta^T}\right)^{-1} \frac{\partial l(\beta)}{\partial \beta^T} \bigg|_{\beta=\beta^{old}} \\
&= \beta^{old} + (X^T W X)^{-1} X^T (y-p(X;\beta^{old})) \\
\end{aligned}
$$

## 实现牛顿法 logistic 回归

In [ ]:
import altair as alt
import numpy as np
import pandas as pd
alt.renderers.enable("svg")

rng = np.random.default_rng(42)
beta_true = np.array([0.5, 1.2, -1])
N_list = [200, 500, 800, 1000]
rounds = 200
tolerance = 1e-5


# X=(1, x1, x2) 其中 x1,x2 ~ N(0,I_N)
def generate_X(N: int):
    X = rng.standard_normal((N, 2))
    X = np.hstack((np.ones((N, 1)), X))
    return X


# y ~ Bernoulli(p) 其中 p = 1/(1+exp(-X^T\beta))
def generate_y(X: np.ndarray, beta: np.ndarray):
    p = 1 / (1 + np.exp(-X @ beta))  # 此处p为向量，np.exp对向量按元素分别计算
    y = rng.binomial(1, p)
    return y


def logistic_regression_newton(
    X: np.ndarray, y: np.ndarray, tolerance: float = 1e-5, max_iter: int = 100
):
    N, p = X.shape
    beta = np.zeros(p)
    for _ in range(max_iter):
        p_vec = 1 / (1 + np.exp(-X @ beta))
        W = np.diag(p_vec * (1 - p_vec))
        # beta_new = beta + (X^TWX)^(-1)X^T(y-p)
        beta_new = beta + np.linalg.inv(X.T @ W @ X) @ X.T @ (y - p_vec)
        if np.max(np.abs(beta_new - beta)) < tolerance:
            return beta_new
        beta = beta_new
    print("exceeded max iterations")
    return beta


def one_round(N: int, beta_true: np.ndarray, tolerance: float = 1e-5):
    X = generate_X(N)
    y_true = generate_y(X, beta_true)
    beta_est = logistic_regression_newton(X, y_true, tolerance)
    return beta_est


def simulate_betas(N_list, beta_true, rounds=200, tolerance=1e-5):
    records = []
    for N in N_list:
        for r in range(rounds):
            beta_est = one_round(N, beta_true, tolerance)
            for i, b in enumerate(beta_est):
                records.append(
                    {
                        "N": N,
                        "run": r,
                        "beta": f"beta_{i}",
                        "beta_hat": b,
                        "beta_true": beta_true[i],
                    }
                )
    df = pd.DataFrame(records)
    df["diff"] = df["beta_hat"] - df["beta_true"]
    return df


def plot_beta_diffs(df, N_list):
    charts = []
    for beta_name in df["beta"].unique():
        df_i = df[df["beta"] == beta_name]
        chart = (
            alt.Chart(df_i)
            .mark_boxplot()
            .encode(
                x=alt.X("N:O", title="Sample Size", sort=N_list),
                y=alt.Y("diff:Q", title=f"{beta_name} (hat - true)"),
                color=alt.Color("N:O", legend=None),
            )
            .properties(title=beta_name)
        )
        charts.append(chart)
    return alt.hconcat(*charts)

betas_df = simulate_betas(N_list, beta_true, rounds)
plot = plot_beta_diffs(betas_df, N_list)
plot

### 1.(2)

已知:
设m+为正样本数, m-为负样本数, D+为正例集合, D-为负例集合
从零点开始, 对于每个ROC曲线上的点(x,y), 减小阈值,下一个样本如果是P, 则坐标为(x,y+1/m+), 如果是N, 则坐标为(x+1/m-,y)

在简单情况下(正负例预测值不相等), ROC曲线呈阶梯状, 按矩形面积求和

对于负例$x_i^-$, 矩形宽为$1/m^-$, 高tpr为在正例中预测值大于$f(x_i^-)$的比例, 即$\frac{1}{m^+}\sum_{j=1}^{m^+} I(f(x_j^+) > f(x_i^-))$
$$
\begin{aligned}
AUC_{ROC} &= \sum_{i=1}^{m^-}\frac{1}{m^-} \frac{1}{m^+}\sum_{j=1}^{m^+} I(f(x_j^+) > f(x_i^-)) \\
&= \frac{1}{m^+ m^-} \sum_{x^+ \in D^+} \sum_{x^- \in D^-} I(f(x^+) > f(x^-)) \\
&= 1-\frac{1}{m^+ m^-} \sum_{x^+ \in D^+} \sum_{x^- \in D^-} I(f(x^+) < f(x^-))\\
&= 1 - l_{rank} \\
&= AUC
\end{aligned}
$$
则有限样本下ROC曲线下方的面积与定义的AUC相等.

In [ ]:
train_data = pd.read_csv("train_data.csv")
test_data = pd.read_csv("test_data.csv")

In [ ]:
print(train_data.describe())
for i in ['room','floor_grp','subway','region','heating']:
    print(f"{i}:")
    print(train_data[i].unique())

In [ ]:
alt.Chart(train_data).mark_bar(binSpacing=0).encode(x=alt.X("rent:Q").bin(maxbins=30), y=alt.Y("count()"))

月租金呈现明显的右偏分布，大多数租金分布在较低区间，少量高价房源拉长右尾。

均值2798.343507
中位数2690.00
最大值6160.00
最小值1150.00

In [ ]:
alt.Chart(train_data).mark_boxplot().encode(
    x=alt.X("region:N").axis(labelAngle=0), y=alt.Y("rent:Q").scale(zero=False)
).properties(width=400)

不同区域月租金差异明显, 东城、朝阳、海淀、西城等区域租金较高, 大兴、房山、通州、顺义等区域租金较低，丰台、昌平、石景山处于中间水平，体现城郊差异。

各区租金分布形态有显著差异，如朝阳区尽管其中位数并不是最高，但离群点较多且数值最高，昌平、通州也存在大量高房租的异常点。这些区的租金分布比较分散；而房山区的箱体较短且无异常值，说明该区域租金较为集中。

In [ ]:
# 用 pandas 进行 one hot encode
X_train = pd.get_dummies(train_data.drop(columns="rent"), drop_first=True, dtype=float)
y_train = train_data["rent"]
X_test = pd.get_dummies(test_data.drop(columns="rent"), drop_first=True, dtype=float)
y_test = test_data["rent"]



In [ ]:
# 使用numpy实现线性回归和岭回归
def linear_regression(X: pd.DataFrame, y: pd.Series):
    """
    X: 自变量, 经过 one hot encode 但未添加截距项
    y: 因变量
    return: 截距项+系数
    """
    X_intecept = np.hstack([np.ones((X.shape[0], 1)), X])
    # \beta = (X^TX)^(-1)X^Ty
    beta = pd.Series(
        np.linalg.inv(X_intecept.T @ X_intecept) @ X_intecept.T @ y,
        index=["intercept"] + list(X.columns),
    )
    return beta


def ridge_regression(X: pd.DataFrame, y: pd.Series, lambda_=0):
    """
    X: 自变量, 经过 one hot encode 但未添加截距项
    y: 因变量
    lambda_: lambda
    return: 截距项+系数
    """
    X_intecept = np.hstack([np.ones((X.shape[0], 1)), X.to_numpy()])
    # \beta_ridge = (X^TX + \lambda I)^(-1)X^Ty
    beta_ridge = (
        np.linalg.inv(X_intecept.T @ X_intecept + lambda_ * np.eye(X_intecept.shape[1]))
        @ X_intecept.T
        @ y
    )
    beta_ridge = pd.Series(beta_ridge)
    beta_ridge.index = ["intercept"] + list(X.columns)

    return beta_ridge


def linear_predict(X: pd.DataFrame, beta: pd.Series):
    """
    X: 自变量, 经过 one hot encode 但未添加截距项
    beta: 截距项+系数
    return: 预测值
    """
    X_intecept = np.hstack([np.ones((X.shape[0], 1)), X])
    y_pred = pd.Series(X_intecept @ beta)
    return y_pred


def reg_mse(y_true: pd.Series, y_pred: pd.Series):
    if len(y_true) != len(y_pred):
        raise ValueError("Length of y_true and y_pred must be the same.")
    mse = ((y_true.to_numpy() - y_pred.to_numpy()) ** 2).mean()
    return mse

In [ ]:
train_data["rent"]**2

In [ ]:
betas = linear_regression(X_train, train_data["rent"])
print(betas)
lin_preds = linear_predict(X_test, betas)
lin_mse = reg_mse(test_data["rent"], lin_preds)

print(f"\nMSE: {lin_mse:}")

In [ ]:
# 为岭回归进行标准化

mean = X_train.mean()
std = X_train.std()
X_train_std = (X_train - mean) / std
X_test_std = (X_test - mean) / std

In [ ]:

from sklearn.model_selection import KFold

def cross_validate_ridge(
    X: pd.DataFrame,
    y: pd.Series,
    lambdas: list,
    n_splits: int = 10,
):
    kf = KFold(n_splits=n_splits, shuffle=False)
    lambda_mse = {lam: [] for lam in lambdas}
    for train_index, val_index in kf.split(X,y):
        X_train_cv, X_val_cv = X.iloc[train_index], X.iloc[val_index]
        y_train_cv, y_val_cv = y.iloc[train_index], y.iloc[val_index]
        for lam in lambdas:
            beta_ridge = ridge_regression(X_train_cv, y_train_cv, lambda_=lam)
            y_val_pred = linear_predict(X_val_cv, beta_ridge)
            if (y_val_cv.index != y_val_pred.index).any():
                print("Index mismatch between y_val_cv and y_val_pred")
            mse = reg_mse(y_val_cv, y_val_pred)
            lambda_mse[lam].append(mse)
            
    avg_lambda_mse = {lam: np.mean(mse_list) for lam, mse_list in lambda_mse.items()}
    return avg_lambda_mse


def plot_ridge_cv_results(avg_mse):
    chart = (
        alt.Chart(
            pd.DataFrame(list(avg_mse.items()), columns=["lambda", "average_mse"])
        )
        .mark_line(point=True)
        .encode(
            x=alt.X(
                "lambda", scale=alt.Scale(type="log"), axis=alt.Axis(title="Lambda")
            ),
            y=alt.Y(
                "average_mse",
                axis=alt.Axis(title="Average MSE"),
                scale=alt.Scale(zero=False),
            ),
            tooltip=["lambda", "average_mse"],
        )
        .properties(title="Ridge Regression Cross-Validation")
    )
    return chart


lambdas = list(np.logspace(-4, -1, 50, base=10))
avg_mse = cross_validate_ridge(X_train_std, train_data["rent"], lambdas)


plot_ridge_cv_results(avg_mse)


取$\lambda = 100$

In [ ]:
ridge_betas = ridge_regression(X_train_std, train_data["rent"], lambda_=0.2)
print(ridge_betas)
ridge_preds = linear_predict(X_test_std, ridge_betas)
ridge_mse = reg_mse(test_data["rent"], ridge_preds)
print(f"\nRidge MSE: {ridge_mse}")

In [ ]:
ridge_betas = ridge_regression(X_train, train_data["rent"], lambda_=0.02)
print(ridge_betas)
ridge_preds = linear_predict(X_test, ridge_betas)
ridge_mse = reg_mse(test_data["rent"], ridge_preds)
print(f"\nRidge MSE: {ridge_mse}")

In [ ]:
def ridge_regression(X: pd.DataFrame, y: pd.Series, lambda_=0):
    """
    X: 自变量, 经过 one hot encode 但未添加截距项
    y: 因变量
    lambda_: lambda
    return: 截距项+系数
    """
    # \beta_ridge = (X^TX + \lambda I)^(-1)X^Ty
    beta_ridge = (
        np.linalg.inv(X.T @ X + lambda_ * np.eye(X.shape[1]))
        @ X.T
        @ y
    )

    beta_ridge.index = X.columns
    return beta_ridge


def linear_predict(X: pd.DataFrame, beta: pd.Series):
    """
    X: 自变量, 经过 one hot encode 但未添加截距项
    beta: 截距项+系数
    return: 预测值
    """
    y_pred = X @ beta
    return pd.Series(y_pred)



from sklearn.model_selection import KFold


def cross_validate_ridge(
    X: pd.DataFrame,
    y: pd.Series|np.ndarray,
    lambdas: list,
    n_splits: int = 10,
):
    # kf = KFold(n_splits=n_splits, shuffle=True, random_state=34)
    kf = KFold(n_splits=n_splits, shuffle=False)
    lambda_mse = {lam: [] for lam in lambdas}
    for train_index, val_index in kf.split(X):
        X_train_cv, X_val_cv = X.iloc[train_index], X.iloc[val_index]
        y_train_cv, y_val_cv = y.iloc[train_index], y.iloc[val_index]
        for lam in lambdas:
            beta_ridge = ridge_regression(X_train_cv, y_train_cv, lambda_=lam)
            y_val_pred = linear_predict(X_val_cv, beta_ridge)
            mse = reg_mse(y_val_cv, y_val_pred)
            lambda_mse[lam].append(mse)
    avg_lambda_mse = {lam: np.mean(mse_list) for lam, mse_list in lambda_mse.items()}
    return avg_lambda_mse


def plot_ridge_cv_results(avg_mse):
    chart = (
        alt.Chart(
            pd.DataFrame(list(avg_mse.items()), columns=["lambda", "average_mse"])
        )
        .mark_line(point=True)
        .encode(
            x=alt.X(
                "lambda", scale=alt.Scale(type="log"), axis=alt.Axis(title="Lambda")
            ),
            y=alt.Y(
                "average_mse",
                axis=alt.Axis(title="Average MSE"),
                scale=alt.Scale(zero=False),
            ),
            tooltip=["lambda", "average_mse"],
        )
        .properties(title="Ridge Regression Cross-Validation")
    )
    return chart


lambdas = list(np.logspace(-3, 0, 50, base=np.e))
avg_mse = cross_validate_ridge(X_train, np.log(train_data["rent"]), lambdas)


plot_ridge_cv_results(avg_mse)



In [ ]:
def ridge_regression(X: pd.DataFrame, y: pd.Series, lambda_=0):
    """
    X: 自变量, 经过 one hot encode 但未添加截距项
    y: 因变量
    lambda_: lambda
    return: 截距项+系数
    """
    # \beta_ridge = (X^TX + \lambda I)^(-1)X^Ty
    beta_ridge = (
        np.linalg.inv(X.T @ X + lambda_ * np.eye(X.shape[1]))
        @ X.T
        @ y
    )

    beta_ridge.index = X.columns
    return beta_ridge


def cross_validate_ridge(
    X: pd.DataFrame,
    y: pd.Series|np.ndarray,
    lambdas: list,
    n_splits: int = 10,
):
    # kf = KFold(n_splits=n_splits, shuffle=True, random_state=34)
    kf = KFold(n_splits=n_splits, shuffle=False)
    lambda_mse = {lam: [] for lam in lambdas}
    for train_index, val_index in kf.split(X):
        X_train_cv, X_val_cv = X.iloc[train_index], X.iloc[val_index]
        y_train_cv, y_val_cv = y.iloc[train_index], y.iloc[val_index]
        for lam in lambdas:
            beta_ridge = ridge_regression(X_train_cv, y_train_cv, lambda_=lam)
            y_val_pred = linear_predict(X_val_cv, beta_ridge)
            mse = reg_mse(y_val_cv, y_val_pred)
            lambda_mse[lam].append(mse)
    avg_lambda_mse = {lam: np.mean(mse_list) for lam, mse_list in lambda_mse.items()}
    return avg_lambda_mse


def plot_ridge_cv_results(avg_mse):
    chart = (
        alt.Chart(
            pd.DataFrame(list(avg_mse.items()), columns=["lambda", "average_mse"])
        )
        .mark_line(point=True)
        .encode(
            x=alt.X(
                "lambda", scale=alt.Scale(type="log"), axis=alt.Axis(title="Lambda")
            ),
            y=alt.Y(
                "average_mse",
                axis=alt.Axis(title="Average MSE"),
                scale=alt.Scale(zero=False),
            ),
            tooltip=["lambda", "average_mse"],
        )
        .properties(title="Ridge Regression Cross-Validation")
    )
    return chart


lambdas = list(np.logspace(-3, 0, 50, base=np.e))
avg_mse = cross_validate_ridge(X_train, np.log(train_data["rent"]), lambdas)


plot_ridge_cv_results(avg_mse)

